# Adding the AF2 Facebook Distillation Dataset

In this example, we demonstrate how to add a custom dataset to DataHub using the AlphaFold2 Facebook distillation dataset. This dataset contains predicted protein structures and associated metadata.

## Dataset Overview

The AF2 Facebook distillation dataset includes:
- Predicted protein structures in CIF format -- these were predicted by AF2
- Metadata for each structure (e.g., sequence length, pLDDT scores)
- MSA files
- Templates

## Steps to Add the Dataset

1. **Load Metadata**: 
   We use pandas to read the metadata from a parquet file, selecting relevant columns.

2. **Create a Custom Parser**:
   We define `AF2FB_DistillationParser`, a custom `MetadataRowParser` that:
   - Constructs file paths for each example that tells us where to find the CIF file
   - Generates standardized example IDs (e.g., `{af2fb}{example_id}{['A']}`, useful for debugging)
   - Provides default values for assembly ID and chain IDs

3. **Initialize Dataset Objects**:
   - Create a `PandasDataset` with the metadata
   - Wrap it in a `StructuralDatasetWrapper` with our custom parser and a `CIFParser`

## Usage

After setting up the dataset, you can use it like any other DataHub dataset.


## Setup
So let's get started by importing the necessary libraries and setting up our environment.

In [ ]:
import pandas as pd
from pathlib import Path

from datahub.datasets.datasets import PandasDataset
# ^-- this is the base dataset class to hold the metadata table

from datahub.datasets.parsers import MetadataRowParser
# ^-- this in abstract class we'll need to implement to say how to assemble the paths from the metadata in table

from datahub.datasets.datasets import StructuralDatasetWrapper
# ^-- this is a wrapper to jointly hold the metadata (which examples we have), the row parser (how to get the data
#     associated with each example), the CIF parser (how to parse the CIF file once we have the path) and the
#     transforms that should be applied to the data once we have it to featurize it.

from cifutils import CIFParser
# ^-- this is the CIF parser for structural data. Don't be deceived by the name, it can parse `.pdb` files too.

In [ ]:
AF2_DISTILLATION_DIR = "/squash/af2_distillation_facebook/"

In [ ]:
# Let's see if we've got the right directory and what to expect there
!ls -l {AF2_DISTILLATION_DIR}
# Let's look at the README file to get an idea of what's in the dataset

In [ ]:
# Let's look at the README file to get an idea of what's in the dataset
!cat {AF2_DISTILLATION_DIR}/README

Next, let's load the metadata table. We will load everything except the sequence column, which is pretty large 
and we don't need it anyways in the metadata, since we have the sequence hash already provided and we could 
always get the sequence back from the cif file.

In [ ]:
df_metadata = pd.read_parquet(
    AF2_DISTILLATION_DIR + "/af2_distillation_facebook.parquet",
    columns=[
        "example_id",
        "original_hash_for_shard",
        "n_atoms",
        "n_res",
        "mean_plddt",
        "min_plddt",
        "median_plddt",
        "sequence_hash",
        "has_msa",
        "msa_depth",
        "has_template",
        "cluster_id",
        # NOTE: The `seq` column is very large, so we don't include it here.
    ],
)

Let's look at the first few rows of the metadata table.

In [ ]:
df_metadata

All good so far -- let's implement the `MetadataRowParser` class. This class provides a standardized way to convert raw data rows into a common dictionary format that can be used by other components of the data processing pipeline.

It has an abstract method `_parse` that we need to implement. This method takes a single row of the metadata table and returns a dictionary that can be used to construct the example (i.e. most notably the path to where to find the structural data for that example).

In [ ]:
class AF2FB_DistillationParser(MetadataRowParser):
    def __init__(self, dataset_dir: str = AF2_DISTILLATION_DIR, file_extension: str = ".cif"):
        self.dataset_dir = Path(dataset_dir)
        self.file_extension = file_extension
        assert self.dataset_dir.exists(), f"Dataset directory {self.dataset_dir} does not exist."

    @staticmethod
    def _get_shard_from_hash(hash_value: str) -> str:
        return f"{hash_value[:2]}/{hash_value[2:4]}/"

    def _parse(self, row: pd.Series) -> dict:
        example_id = row["example_id"]
        sequence_hash = row["sequence_hash"]

        path = (
            self.dataset_dir / "cif" / self._get_shard_from_hash(sequence_hash) / f"{example_id}{self.file_extension}"
        )

        return {
            "example_id": example_id,
            "path": path,
            "assembly_id": "1",  # just default to the first assembly (=identity if none given)
            # "query_pn_unit_iids": ["A_1"],  # just default to the first and only chain (does not need to be given)
            "sequence_hash": sequence_hash,
        }

Finally, let's wrap it up in a `StructuralDatasetWrapper` and we're done.

In [ ]:
af2fb_metadata = PandasDataset(data=df_metadata, name="af2_distillation_facebook", id_column="example_id")

af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    dataset_parser=AF2FB_DistillationParser(),  # <-- how to take the metadata and extract a path to the relevant files
    cif_parser=CIFParser(),  # <-- how to parse the structural file once we have the path (a computationally predicted CIF in this case)
    cif_parser_args={"assume_residues_all_resolved": True},  # <-- any additional arguments to pass to the CIF parser,
    # here we are telling the CIF parser to assume that all the residues are resolved, which is the setting that's required
    # when loading computationally predicted structures, for which there are no missing atoms or residues.
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
)

Let's see if we can load an example.

In [ ]:
af2fb_dataset[0]

We can also load by example ID.

In [ ]:
af2fb_dataset.id_to_idx("UniRef50_A0A1S3ZVX8")

In [ ]:
af2fb_dataset[af2fb_dataset.id_to_idx("UniRef50_A0A1S3ZVX8")]

Finally, let's add a few transforms, just for fun

In [ ]:
from datahub.transforms.base import Compose
from datahub.transforms.atom_array import AddGlobalAtomIdAnnotation
from datahub.transforms.atomize import AtomizeByCCDName
from datahub.encoding_definitions import RF2_ATOM36_ENCODING
from datahub.transforms.encoding import EncodeAtomArray
from datahub.transforms.crop import CropSpatialLikeAF3

af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    dataset_parser=AF2FB_DistillationParser(),  # <-- how to take the metadata and extract a path to the relevant files
    cif_parser=CIFParser(),  # <-- how to parse the structural file once we have the path (a computationally predicted CIF in this case)
    cif_parser_args={"assume_residues_all_resolved": True},  # <-- any additional arguments to pass to the CIF parser,
    # here we are telling the CIF parser to assume that all the residues are resolved, which is the setting that's required
    # when loading computationally predicted structures, for which there are no missing atoms or residues.
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
    # These transforms will be applied to the data once loaded and the result will be returned
    transform=Compose(
        [
            AddGlobalAtomIdAnnotation(),
            AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=RF2_ATOM36_ENCODING.tokens),
            EncodeAtomArray(
                encoding=RF2_ATOM36_ENCODING,
            ),
            CropSpatialLikeAF3(crop_size=128),
        ]
    ),
)
af2fb_dataset[0]

Oops! I did something stupid here -- I tried to first encode and then crop. That's dangerous because it could be that we 
crop mid-encoded-token or so (and it's also inefficient for large structures, because I only need to encode what's in the crop
since that's everything I'm gonna use anyways.). 
So the pipeline does not allow us to do that errors with a `TransformPipelineError`.

Let's fix it by first cropping and then encoding.

In [ ]:
af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    dataset_parser=AF2FB_DistillationParser(),  # <-- how to take the metadata and extract a path to the relevant files
    cif_parser=CIFParser(),  # <-- how to parse the structural file once we have the path (a computationally predicted CIF in this case)
    cif_parser_args={"assume_residues_all_resolved": True},  # <-- any additional arguments to pass to the CIF parser,
    # here we are telling the CIF parser to assume that all the residues are resolved, which is the setting that's required
    # when loading computationally predicted structures, for which there are no missing atoms or residues.
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
    # These transforms will be applied to the data once loaded and the result will be returned
    transform=Compose(
        [
            AddGlobalAtomIdAnnotation(),
            AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=RF2_ATOM36_ENCODING.tokens),
            CropSpatialLikeAF3(crop_size=128),
            EncodeAtomArray(
                encoding=RF2_ATOM36_ENCODING,
            ),
        ]
    ),
)
data = af2fb_dataset[0]

Let's take a peek at our crop:

In [ ]:
from cifutils.utils.visualize import view

view(data["atom_array"])